In [1]:
import sys
sys.path.append(f"./../")

import matplotlib.pyplot as plt
import numpy as np
import os
import networkx as nx
import gc
import psutil
from datetime import datetime
from contextlib import contextmanager
import itertools
from collections import defaultdict

from src.graphs import StaticGraph, IntersectingEdgesGraph, MultiEdgeGraph
from src.misc import is_bipartite
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, SparsePauliOp, Pauli
from qiskit.circuit.library import PauliEvolutionGate
from scipy.linalg import expm

In [2]:
def get_memory_usage():
    """Get current memory usage in MB."""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def log_progress(message, log_file=None):
    """Log progress with timestamp and memory usage."""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    mem_usage = get_memory_usage()
    log_msg = f"[{timestamp}] {message}"
    print(log_msg)
    if log_file:
        with open(log_file, 'a') as f:
            f.write(log_msg + '\n')

@contextmanager
def memory_cleanup():
    """Context manager for memory cleanup."""
    try:
        yield
    finally:
        gc.collect()


In [3]:
def graph_to_bitstring_edges(graph):
    """Convert graph to bitstring edges."""
    num_nodes = len(graph.nodes)
    num_bits = len(bin(num_nodes - 1)) - 2
    node_to_bitstring = {node: format(node, f'0{num_bits}b') for node in graph.nodes}
    edges_bitstring = {(node_to_bitstring[u], node_to_bitstring[v]) for u, v in graph.edges}
    del node_to_bitstring
    return edges_bitstring

def count_gates_matching_dynamic_walk(edges, n_steps=1):
    """Count gates for matching dynamic walk."""
    log_progress("Computing Matching Dynamic walk")
    
    with memory_cleanup():
        try:
            static_G = StaticGraph(edges)
            intersecting_G = IntersectingEdgesGraph(edges)
            
            big_qc = QuantumCircuit(static_G.n_qubits)
            for _ in range(n_steps):
                for subgraph in intersecting_G.subgraphs:
                    with memory_cleanup():
                        G = MultiEdgeGraph(subgraph.edges)
                        sub_qc = G.get_qc(simplified=True)
                        big_qc = big_qc.compose(sub_qc)
                        del G, sub_qc
            
            transpiled_qc = transpile(big_qc, basis_gates=['cx', 'u3'], 
                                    optimization_level=3)
            gate_counts = transpiled_qc.count_ops()
            
            cx_count = gate_counts.get('cx', 0)
            u3_count = gate_counts.get('u3', 0)
            depth = transpiled_qc.depth()
            
            log_progress(f"Matching Dynamic - CX: {cx_count}, U3: {u3_count}, Depth: {depth}")
            return cx_count, u3_count, depth
            
        except Exception as e:
            log_progress(f"Error in Matching Dynamic walk: {str(e)}")
            return None

def count_gates_exact_walk(edges, delta_t):
    """Exact quantum walk implementation."""
    log_progress("Computing Exact walk")
    
    with memory_cleanup():
        try:
            static_G = StaticGraph(edges)
            H = -1j * delta_t * static_G.get_adj_mat()
            U = Operator(expm(H))
            
            qc = QuantumCircuit(static_G.n_qubits)
            qc.unitary(U, range(static_G.n_qubits), label='ExactWalk')
            
            transpiled_qc = transpile(qc, basis_gates=['cx', 'u3'],
                                    optimization_level=3)
            gate_counts = transpiled_qc.count_ops()
            
            cx_count = gate_counts.get('cx', 0)
            u3_count = gate_counts.get('u3', 0)
            depth = transpiled_qc.depth()
            
            log_progress(f"Exact - CX: {cx_count}, U3: {u3_count}, Depth: {depth}")
            return cx_count, u3_count, depth
            
        except Exception as e:
            log_progress(f"Error in Exact walk: {str(e)}")
            return None

def count_gates_pauli_decomp(edges, delta_t):
    """Count gates for Pauli decomposition with proper Pauli evolution."""
    log_progress("Computing Pauli decomposition")
    
    with memory_cleanup():
        try:
            # Construct the adjacency matrix
            static_G = StaticGraph(edges)
            H = static_G.get_adj_mat()
            n = static_G.n_qubits
            
            # Decompose the Hamiltonian into Pauli basis
            pauli_strings = []
            real_coeffs = []
            imag_coeffs = []
            for pauli_string in [''.join(p) for p in itertools.product('IXYZ', repeat=n)]:
                P = Pauli(pauli_string)
                P_op = Operator(P).data
                coeff = np.trace(P_op.conj().T @ H) / (2**n)
                if not np.isclose(coeff, 0, atol=1e-10):
                    pauli_strings.append(pauli_string)
                    real_coeffs.append(float(np.real(coeff)))
                    imag_coeffs.append(float(np.imag(coeff)))
            
            # Create separate SparsePauliOps for real and imaginary parts
            qc = QuantumCircuit(n)
            
            # Real part evolution
            if any(c != 0 for c in real_coeffs):
                real_pauli_op = SparsePauliOp(pauli_strings, real_coeffs)
                real_evo_gate = PauliEvolutionGate(real_pauli_op, time=-delta_t)
                qc.append(real_evo_gate, range(n))
            
            # Imaginary part evolution
            if any(c != 0 for c in imag_coeffs):
                # Treat imaginary evolution as real evolution with adjusted coefficients
                imag_pauli_op = SparsePauliOp(pauli_strings, imag_coeffs)
                imag_evo_gate = PauliEvolutionGate(imag_pauli_op, time=-delta_t)
                qc.append(imag_evo_gate, range(n))
            
            # Transpile the circuit to target basis gates
            transpiled_qc = transpile(qc, basis_gates=['cx', 'u3'], optimization_level=3)
            
            # Count gates
            gate_counts = transpiled_qc.count_ops()
            cx_count = gate_counts.get('cx', 0)
            u3_count = gate_counts.get('u3', 0)
            depth = transpiled_qc.depth()
            
            log_progress(f"Pauli - CX: {cx_count}, U3: {u3_count}, Depth: {depth}")
            return cx_count, u3_count, depth
        
        except Exception as e:
            log_progress(f"Error in Pauli decomposition: {str(e)}")
            return None


In [4]:
def process_single_graph_multi_method(edges, index, delta_t, n_steps):
    """Process a single graph comparing matching vs exact and Pauli methods."""
    try:
        with memory_cleanup():
            log_progress(f"\nProcessing graph {index}")
            
            # Get matching dynamic counts
            dynamic_counts = count_gates_matching_dynamic_walk(edges, n_steps=n_steps)
            if not dynamic_counts:
                return None
            dynamic_cx, dynamic_u3, dynamic_depth = dynamic_counts
            
            # Get exact counts
            exact_counts = count_gates_exact_walk(edges, delta_t)
            if not exact_counts:
                return None
            exact_cx, exact_u3, exact_depth = exact_counts
            
            # Get Pauli counts
            pauli_counts = count_gates_pauli_decomp(edges, delta_t)
            if not pauli_counts:
                return None
            pauli_cx, pauli_u3, pauli_depth = pauli_counts
            
            # Calculate differences
            exact_cx_diff = dynamic_cx - exact_cx
            exact_u3_diff = dynamic_u3 - exact_u3
            pauli_cx_diff = dynamic_cx - pauli_cx
            pauli_u3_diff = dynamic_u3 - pauli_u3
            
            # Determine results based on CX gates (primary metric)
            exact_result = "win" if exact_cx_diff < 0 else "lose" if exact_cx_diff > 0 else "draw"
            pauli_result = "win" if pauli_cx_diff < 0 else "lose" if pauli_cx_diff > 0 else "draw"
            
            # Determine category
            category = None
            if exact_result == "win" and pauli_result == "win":
                category = "win_both"
            elif exact_result == "lose" and pauli_result == "lose":
                category = "lose_both"
            elif exact_result == "draw" and pauli_result == "draw":
                category = "draw_both"
            elif exact_result == "win" and pauli_result == "lose":
                category = "win_exact_lose_pauli"
            elif exact_result == "lose" and pauli_result == "win":
                category = "win_pauli_lose_exact"
            else:
                category = "other"
            
            # Log detailed results including category
            log_progress(f"Graph {index} Results:")
            log_progress(f"  Category: {category}")
            log_progress(f"  Matching vs Exact: {exact_result} (CX diff: {exact_cx_diff}, U3 diff: {exact_u3_diff})")
            log_progress(f"  Matching vs Pauli: {pauli_result} (CX diff: {pauli_cx_diff}, U3 diff: {pauli_u3_diff})")
            
            # Check bipartiteness
            G = nx.Graph(edges)
            is_bip = is_bipartite(G)
            log_progress(f"  Bipartite: {is_bip}")
            
            return {
                'index': index,
                'edges': edges,
                'category': category,
                'exact_result': exact_result,
                'pauli_result': pauli_result,
                'is_bipartite': is_bip,
                'dynamic_cx': dynamic_cx,
                'dynamic_u3': dynamic_u3,
                'exact_cx': exact_cx,
                'exact_u3': exact_u3,
                'pauli_cx': pauli_cx,
                'pauli_u3': pauli_u3,
                'exact_cx_diff': exact_cx_diff,
                'exact_u3_diff': exact_u3_diff,
                'pauli_cx_diff': pauli_cx_diff,
                'pauli_u3_diff': pauli_u3_diff
            }
            
    except Exception as e:
        log_progress(f"Error processing graph {index}: {str(e)}")
        return None

def categorize_results(results):
    """Categorize graphs based on comparison results."""
    categories = {
        'win_both': [],
        'lose_both': [],
        'draw_both': [],
        'win_exact_lose_pauli': [],
        'win_pauli_lose_exact': [],
        'other': []
    }
    
    for result in results:
        if result['exact_result'] == "win" and result['pauli_result'] == "win":
            categories['win_both'].append(result)
        elif result['exact_result'] == "lose" and result['pauli_result'] == "lose":
            categories['lose_both'].append(result)
        elif result['exact_result'] == "draw" and result['pauli_result'] == "draw":
            categories['draw_both'].append(result)
        elif result['exact_result'] == "win" and result['pauli_result'] == "lose":
            categories['win_exact_lose_pauli'].append(result)
        elif result['exact_result'] == "lose" and result['pauli_result'] == "win":
            categories['win_pauli_lose_exact'].append(result)
        else:
            categories['other'].append(result)
    
    return categories

def save_categorized_graphs(categories, n_vertex):
    """Save graphs in each category to separate G6 files."""
    os.makedirs('../data/graphs', exist_ok=True)
    
    for category, graphs in categories.items():
        if graphs:
            filename = f'../data/graphs/{category}_{n_vertex}v.g6'
            log_progress(f"Saving {len(graphs)} graphs to {filename}")
            with open(filename, 'w') as f:
                for graph in graphs:
                    G = nx.Graph(graph['edges'])
                    g6_string = nx.to_graph6_bytes(G, header=False).decode().strip()
                    f.write(f"{g6_string}\n")

def calculate_statistics(categories):
    """Calculate statistics for each category and bipartite percentages."""
    stats = {}
    total_graphs = sum(len(graphs) for graphs in categories.values())
    
    for category, graphs in categories.items():
        if graphs:
            percentage = (len(graphs) / total_graphs) * 100
            if category == 'win_both':
                bipartite_graphs = [g for g in graphs if g['is_bipartite']]
                bip_percentage = (len(bipartite_graphs) / len(graphs)) * 100
                stats[category] = {
                    'count': len(graphs),
                    'percentage': percentage,
                    'bipartite_percentage': bip_percentage
                }
            else:
                stats[category] = {
                    'count': len(graphs),
                    'percentage': percentage
                }
    
    return stats

def plot_results(categories, n_vertex):
    """Create visualization plots for the results."""
    os.makedirs('../data/plots', exist_ok=True)
    
    # Pie chart of categories
    plt.figure(figsize=(12, 8))
    labels = []
    sizes = []
    colors = ['green', 'red', 'blue', 'yellow', 'purple', 'gray']
    
    total = sum(len(graphs) for graphs in categories.values())
    for (category, graphs), color in zip(categories.items(), colors):
        if graphs:
            percentage = len(graphs) / total * 100
            labels.append(f'{category}\n({len(graphs)}, {percentage:.1f}%)')
            sizes.append(len(graphs))
    
    plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%')
    plt.title('Distribution of Graph Categories')
    plt.savefig(f'../data/plots/categories_pie_{n_vertex}v.png')
    plt.show()

def process_graphs_multi_method(g6_file, n_vertex, delta_t, n_steps, batch_size=10):
    """Process all graphs and compare methods."""
    results = []
    error_count = 0
    
    log_progress(f"Processing graphs from {g6_file}")
    log_progress(f"Parameters: delta_t={delta_t}, n_steps={n_steps}, batch_size={batch_size}")
    
    with open(g6_file, 'r') as file:
        for i, line in enumerate(file):
            if i % batch_size == 0:
                log_progress(f"\nProcessing batch starting at graph {i}")
            
            try:
                graph = nx.from_graph6_bytes(line.strip().encode('utf-8'))
                edges = graph_to_bitstring_edges(graph)
                
                result = process_single_graph_multi_method(edges, i, delta_t, n_steps)
                if result:
                    results.append(result)
                else:
                    error_count += 1
                    
            except Exception as e:
                error_count += 1
                log_progress(f"Error processing graph {i}: {str(e)}")
                continue
    
    # Categorize and analyze results
    categories = categorize_results(results)
    save_categorized_graphs(categories, n_vertex)
    
    # Calculate and display statistics
    stats = calculate_statistics(categories)
    log_progress("\nFinal Results Summary:")
    for category, stat in stats.items():
        log_msg = f"{category}: {stat['count']} ({stat['percentage']:.1f}%)"
        if 'bipartite_percentage' in stat:
            log_msg += f" - Bipartite: {stat['bipartite_percentage']}%"
        log_progress(log_msg)
    
    # Create visualizations
    plot_results(categories, n_vertex)
    
    return results, categories, stats

def run_single_analysis(n_vertex, delta_t, n_steps, batch_size):
    """Run a single complete analysis and return the results."""
    g6_file = f'../data/graphs/graph{n_vertex}c.g6'
    return process_graphs_multi_method(g6_file, n_vertex, delta_t, n_steps, batch_size)

def aggregate_statistics(all_stats):
    """Aggregate statistics across multiple runs."""
    agg_stats = defaultdict(lambda: defaultdict(list))
    
    for run_stats in all_stats:
        for category, stat in run_stats.items():
            agg_stats[category]['count'].append(stat['count'])
            agg_stats[category]['percentage'].append(stat['percentage'])
            if 'bipartite_percentage' in stat:
                agg_stats[category]['bipartite_percentage'].append(stat['bipartite_percentage'])
    
    # Calculate means and standard deviations
    final_stats = {}
    for category, stats in agg_stats.items():
        final_stats[category] = {
            'count_mean': np.mean(stats['count']),
            'count_std': np.std(stats['count']),
            'percentage_mean': np.mean(stats['percentage']),
            'percentage_std': np.std(stats['percentage'])
        }
        if 'bipartite_percentage' in stats:
            final_stats[category]['bipartite_percentage_mean'] = np.mean(stats['bipartite_percentage'])
            final_stats[category]['bipartite_percentage_std'] = np.std(stats['bipartite_percentage'])
    
    return final_stats


In [5]:
# Parameters
n_qubits = 3
n_vertex = 2**n_qubits
T = 0.1
delta_t = 0.1
n_steps = int(T/delta_t)
batch_size = 2
n_runs = 3  # Number of runs

In [ ]:
# Input file
g6_file = f'../data/graphs/graph{n_vertex}c.g6'

# Create output directories
os.makedirs('../data/graphs', exist_ok=True)
os.makedirs('../data/plots', exist_ok=True)

# Log starting parameters
log_progress("\nStarting analysis with parameters:")
log_progress(f"Number of qubits: {n_qubits}")
log_progress(f"Number of vertices: {n_vertex}")
log_progress(f"Evolution time T: {T}")
log_progress(f"Time step delta_t: {delta_t}")
log_progress(f"Number of steps: {n_steps}")
log_progress(f"Batch size: {batch_size}")

# Log starting parameters
log_progress(f"Number of runs: {n_runs}")

# Store results from all runs
all_results = []
all_categories = []
all_stats = []

# Perform multiple runs
for run in range(n_runs):
    log_progress(f"\nStarting run {run + 1}/{n_runs}")
    results, categories, stats = run_single_analysis(n_vertex, delta_t, n_steps, batch_size)
    all_results.append(results)
    all_categories.append(categories)
    all_stats.append(stats)
    log_progress(f"Completed run {run + 1}")

# Aggregate statistics across runs
aggregated_stats = aggregate_statistics(all_stats)

# Final summary
log_progress(f"\nFinal Results Summary (across all {n_runs} runs):")

# Detailed category analysis
log_progress("\nDetailed Category Analysis (mean ± std):")
for category, stat in aggregated_stats.items():
    log_msg = (f"{category}:\n"
                f"  Count: {stat['count_mean']:.1f} ± {stat['count_std']:.1f}\n"
                f"  Percentage: {stat['percentage_mean']:.1f}% ± {stat['percentage_std']:.1f}%")
    if 'bipartite_percentage_mean' in stat:
        log_msg += (f"\n  Bipartite: {stat['bipartite_percentage_mean']:.1f}% ± "
                    f"{stat['bipartite_percentage_std']:.1f}%")
    log_progress(log_msg)

# Save final results from the last run (or you could choose to save from all runs)
save_categorized_graphs(all_categories[-1], n_vertex)
plot_results(all_categories[-1], n_vertex)

[2025-01-01 04:54:13] 
Starting analysis with parameters:
[2025-01-01 04:54:13] Number of qubits: 3
[2025-01-01 04:54:13] Number of vertices: 8
[2025-01-01 04:54:13] Evolution time T: 0.1
[2025-01-01 04:54:13] Time step delta_t: 0.1
[2025-01-01 04:54:13] Number of steps: 1
[2025-01-01 04:54:13] Batch size: 2
[2025-01-01 04:54:13] Number of runs: 3
[2025-01-01 04:54:13] 
Starting run 1/3
[2025-01-01 04:54:13] Processing graphs from ../data/graphs/graph8c.g6
[2025-01-01 04:54:13] Parameters: delta_t=0.1, n_steps=1, batch_size=2
[2025-01-01 04:54:13] 
Processing batch starting at graph 0
[2025-01-01 04:54:13] 
Processing graph 0
[2025-01-01 04:54:13] Computing Matching Dynamic walk
[2025-01-01 04:54:14] Matching Dynamic - CX: 59, U3: 60, Depth: 93
[2025-01-01 04:54:14] Computing Exact walk
[2025-01-01 04:54:14] Exact - CX: 20, U3: 37, Depth: 41
[2025-01-01 04:54:14] Computing Pauli decomposition
[2025-01-01 04:54:14] Pauli - CX: 30, U3: 55, Depth: 58
[2025-01-01 04:54:14] Graph 0 Results: